# BiLSTM Ablation — POS / DEP / SHAPE

We train a BiLSTM fake-news classifier on every combination of the three
tag-sequence features and save each trained model so the evaluation and
ablation analysis can be run separately.

| # | Features |
|---|----------|
| 1 | POS |
| 2 | DEP |
| 3 | SHAPE |
| 4 | POS + DEP |
| 5 | POS + SHAPE |
| 6 | DEP + SHAPE |
| 7 | POS + DEP + SHAPE |

A single stratified train/val/test split is fixed up front and saved, so all
seven models are trained and tested on exactly the same data.

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
SEED = 404
DATA_PATH = "Dataset + Feature/sequences.parquet"
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

MAX_LEN = 160          # truncate long articles (keeps CPU training tractable)
EMB_DIM = 48
HIDDEN_DIM = 64
DROPOUT = 0.3
BATCH_SIZE = 128
EPOCHS = 6
LR = 1e-3
PATIENCE = 2           # early stopping on val F1

CHANNELS = ["pos", "dep", "shape"]
COL = {"pos": "pos_ids", "dep": "dep_ids", "shape": "shape_ids"}
PAD_IDX = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
df = pd.read_parquet(DATA_PATH)
print(f"{len(df):,} articles   label balance: {df['label'].value_counts().to_dict()}")

# Vocab size per channel = highest tag id + 1 (pad=0 already included).
vocab_sizes = {ch: int(max(seq.max() for seq in df[COL[ch]])) + 1 for ch in CHANNELS}
print("vocab sizes:", vocab_sizes)

## Split

80 / 10 / 10 stratified on the label, fixed by `SEED`. Indices are saved to
`models/split.json` so evaluation uses the identical test set.

In [ ]:
idx = np.arange(len(df))
train_idx, temp_idx = train_test_split(
    idx, test_size=0.2, random_state=SEED, stratify=df["label"]
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, random_state=SEED, stratify=df["label"].iloc[temp_idx]
)
print(f"train {len(train_idx)}  val {len(val_idx)}  test {len(test_idx)}")

(MODEL_DIR / "split.json").write_text(json.dumps({
    "seed": SEED,
    "train": train_idx.tolist(),
    "val": val_idx.tolist(),
    "test": test_idx.tolist(),
}))

In [ ]:
class TagDataset(Dataset):
    def __init__(self, frame, channels):
        self.channels = channels
        self.data = {
            ch: [np.array(s[:MAX_LEN], dtype=np.int64) for s in frame[COL[ch]]]
            for ch in channels
        }
        self.labels = frame["label"].to_numpy(dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        seqs = {ch: torch.from_numpy(self.data[ch][i]) for ch in self.channels}
        length = len(seqs[self.channels[0]])
        return seqs, length, self.labels[i]


def make_collate(channels):
    def collate(batch):
        seqs, lengths, labels = zip(*batch)
        padded = {
            ch: pad_sequence([s[ch] for s in seqs], batch_first=True,
                             padding_value=PAD_IDX)
            for ch in channels
        }
        return (padded,
                torch.tensor(lengths),
                torch.tensor(labels))
    return collate

In [ ]:
class BiLSTMTagger(nn.Module):
    """One embedding per active channel, concatenated then run through a BiLSTM
    and mean-pooled over the real tokens."""

    def __init__(self, vocab_sizes_active):
        super().__init__()
        self.channels = list(vocab_sizes_active.keys())
        self.embeddings = nn.ModuleDict({
            ch: nn.Embedding(size, EMB_DIM, padding_idx=PAD_IDX)
            for ch, size in vocab_sizes_active.items()
        })
        self.lstm = nn.LSTM(EMB_DIM * len(self.channels), HIDDEN_DIM,
                            batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(HIDDEN_DIM * 2, 2)

    def forward(self, batch, lengths):
        embs = [self.embeddings[ch](batch[ch]) for ch in self.channels]
        x = torch.cat(embs, dim=-1) if len(embs) > 1 else embs[0]

        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True,
                                      enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)

        mask = (torch.arange(out.size(1), device=out.device)[None, :]
                < lengths.to(out.device)[:, None]).unsqueeze(-1)
        pooled = (out * mask).sum(1) / lengths.to(out.device).clamp(min=1)[:, None]
        return self.fc(self.dropout(pooled))

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss, preds, golds = 0.0, [], []

    with torch.set_grad_enabled(train):
        for batch, lengths, labels in loader:
            batch = {ch: t.to(device) for ch, t in batch.items()}
            lengths, labels = lengths.to(device), labels.to(device)

            logits = model(batch, lengths)
            loss = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(labels)
            preds.extend(logits.argmax(1).cpu().tolist())
            golds.extend(labels.cpu().tolist())

    return (total_loss / len(golds),
            accuracy_score(golds, preds),
            f1_score(golds, preds))

In [ ]:
def train_combo(name, channels):
    print(f"\n=== {name}  ({' + '.join(channels)}) ===")
    vsz = {ch: vocab_sizes[ch] for ch in channels}

    sets = {
        split: TagDataset(df.iloc[ix], channels)
        for split, ix in [("train", train_idx), ("val", val_idx)]
    }
    collate = make_collate(channels)
    loaders = {
        split: DataLoader(ds, batch_size=BATCH_SIZE,
                          shuffle=(split == "train"), collate_fn=collate)
        for split, ds in sets.items()
    }

    model = BiLSTMTagger(vsz).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_f1, best_state, stale = -1.0, None, 0
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc, tr_f1 = run_epoch(model, loaders["train"], criterion, optimizer)
        vl_loss, vl_acc, vl_f1 = run_epoch(model, loaders["val"], criterion)
        print(f"  epoch {epoch}  train f1 {tr_f1:.4f}  |  val acc {vl_acc:.4f}  f1 {vl_f1:.4f}")

        if vl_f1 > best_f1:
            best_f1 = vl_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f"  early stop (best val f1 {best_f1:.4f})")
                break

    ckpt = {
        "name": name,
        "channels": channels,
        "vocab_sizes": vsz,
        "hparams": {"emb_dim": EMB_DIM, "hidden_dim": HIDDEN_DIM,
                    "dropout": DROPOUT, "max_len": MAX_LEN},
        "best_val_f1": best_f1,
        "state_dict": best_state,
    }
    path = MODEL_DIR / f"bilstm_{name}.pt"
    torch.save(ckpt, path)
    print(f"  saved -> {path}  (best val f1 {best_f1:.4f})")
    return best_f1

## Train all seven combinations

In [ ]:
COMBOS = {
    "pos":            ["pos"],
    "dep":            ["dep"],
    "shape":          ["shape"],
    "pos_dep":        ["pos", "dep"],
    "pos_shape":      ["pos", "shape"],
    "dep_shape":      ["dep", "shape"],
    "pos_dep_shape":  ["pos", "dep", "shape"],
}

results = {name: train_combo(name, chans) for name, chans in COMBOS.items()}

In [ ]:
print("Best validation F1 per model:")
for name, f1 in sorted(results.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {name:<16} {f1:.4f}")

## Hand-off

`models/` now contains seven `bilstm_<combo>.pt` checkpoints and `split.json`.

Each checkpoint stores `channels`, `vocab_sizes`, `hparams` and the `state_dict`,
so a model can be rebuilt and loaded for evaluation without re-reading any
training config. Evaluate on the saved `test` indices from `split.json` to keep
the comparison fair across the ablation.